# Layer 1 — Copy Risk Checker

Flags risk patterns in journalism prose or AI-generated text.  
**Does not** decide truth, rewrite copy, or score quality.  
Every flag is produced by a named, auditable rule.

Output: colored inline highlights + flag table.

## 1. Dependencies

In [13]:
import re
import spacy
from dataclasses import dataclass
from IPython.display import HTML, display

nlp = spacy.load("en_core_web_sm")

## 2. Flag type definitions

Each flag has a type, priority, and one-line reason.  
Priority: **High** = must verify before publish, **Medium** = should verify, **Low** = good to check.

In [14]:
@dataclass
class Flag:
    start: int        # char offset in original text
    end: int
    text: str         # matched span
    flag_type: str
    priority: str     # High / Medium / Low
    reason: str

# Colors per flag TYPE — each risk category gets a distinct color
FLAG_COLORS = {
    "quantitative_claim":  "#74c0fc",  # blue    — numbers need sources
    "vague_attribution":   "#ff6b6b",  # red     — who said this?
    "passive_attribution": "#f783ac",  # rose    — actor removed entirely
    "causal_claim":        "#ff922b",  # orange  — X caused Y
    "certainty_language":  "#ffd43b",  # yellow  — unhedged assertion
    "trend_language":      "#63e6be",  # teal    — magnitude without evidence
    "comparative_claim":   "#a9e34b",  # green   — compared to what?
    "temporal_claim":      "#ffa8a8",  # pink    — verify the timeframe
    "named_entity":        "#dee2e6",  # grey    — verify name/role
}

PRIORITY_RANK = {"High": 0, "Medium": 1, "Low": 2}

## 3. Pattern definitions

### Tier 1 — Regex / lexicon (no model, fully deterministic)

In [15]:
# Each entry: (flag_type, priority, reason, compiled_regex)
REGEX_PATTERNS = [

    # Hedged numbers — "roughly $72,000", "nearly half", "approximately 400"
    # Must come BEFORE the base quantitative_claim entry so it wins deduplication
    # on overlapping spans (hedge span starts earlier, covers the qualifier)
    (
        "quantitative_claim", "High",
        "Hedged figure — does the reporter have the exact number?",
        re.compile(
            r"""(?x)
            \b(?:nearly|roughly|approximately|about|around|almost|
               an?\s+estimated|more\s+or\s+less|upwards?\s+of|
               as\s+(?:many|few|much)\s+as)
            \s+
            (?:
                \d+(?:\.\d+)?%                                      # nearly 30%
              | \$\d+(?:[,.]\d+)*(?:\s*(?:million|billion|trillion|thousand))?  # roughly $72,000
              | \d+(?:,\d{3})+                                      # about 1,200
              | \d+(?:\.\d+)?\s*(?:million|billion|trillion|thousand)  # approximately 2 million
              | \d+(?:\.\d+)?\s+cents?                              # almost 9 cents
              | \d+\s+(?:people|jobs?|homes?|cases?|deaths?|workers?|residents?|students?)
              | half | a\s+(?:third|quarter|fifth)                  # nearly half, about a third
            )
            """,
            re.IGNORECASE
        )
    ),

    # Numbers: percentages, dollar amounts, large counts, ordinals, bare integers
    (
        "quantitative_claim", "High",
        "Specific number — source needed",
        re.compile(
            r"""(?x)
            \b\d+(?:\.\d+)?%                                        # percentages: 27%, 3.5%
            | \$\d+(?:[,.]\d+)*(?:\s*(?:million|billion|trillion|thousand))?  # $4.2 million, $72,000
            | \b\d+(?:,\d{3})+\b                                    # formatted: 1,200
            | \b\d+(?:\.\d+)?\s*(?:million|billion|trillion|thousand)\b  # 2.1 million, 50 thousand
            | \b\d+(?:\.\d+)?\s+cents?\b                            # 8.8 cents
            | \branked?\s+\d+(?:st|nd|rd|th)?\b                     # ranked 3rd
            | \b\d+\s+(?:newly\s+)?(?:wallets?|accounts?|users?|addresses?)\b  # 50 wallets
            | \b(?:one|two|three|four|five|six|seven|eight|nine|ten)\s+(?:million|billion|thousand)\b
            """,
            re.IGNORECASE
        )
    ),

    # Vague attribution — high risk because it obscures the source
    (
        "vague_attribution", "High",
        "Unattributed source — who specifically?",
        re.compile(
            r"""(?x)
            \b(?:experts?|officials?|researchers?|scientists?|analysts?|sources?|
               investigators?|authorities|critics?|observers?|insiders?|advocates?)
            \s+(?:say|says|said|claim|claims|claimed|warn|warns|warned|
                 argue|argues|argued|suggest|suggests|suggested|report|reports|reported|
                 found|find|finds)
            | (?:studies|research|data|reports?|evidence|findings?)\s+(?:show|shows|suggest|indicate|find|found)
            | \baccording\s+to\s+(?:sources?|officials?|experts?|reports?)\b
            | \bmany\s+(?:believe|say|argue|think|feel)\b
            | \bsome\s+(?:believe|say|argue|think|suggest)\b
            """,
            re.IGNORECASE
        )
    ),

    # Passive attribution — actor removed entirely; common in AI-generated text
    # "it was found that", "it has been reported", "it is estimated that"
    # Different from vague_attribution (which has a noun like "experts") —
    # here the subject is a dummy "it" and the real actor is completely hidden.
    (
        "passive_attribution", "High",
        "Actor removed — who found/reported/estimated this?",
        re.compile(
            r"""(?x)
            \bit\s+(?:has\s+been|have\s+been|was|were|is|are)\s+
            (?:found|reported|estimated|suggested|noted|observed|
               believed|claimed|alleged|determined|confirmed|shown|
               established|documented|revealed|understood|acknowledged)
            (?:\s+that)?
            | \bit\s+(?:appears?|seems?|looks?)\s+(?:that\s+)?(?:the\s+)?(?:data\s+)?(?:suggests?|shows?|indicates?)
            | \b(?:is|are|was|were)\s+(?:widely\s+)?(?:believed|reported|understood|considered|known)\s+to\b
            | \bhas\s+been\s+(?:widely\s+)?(?:reported|noted|documented|established|confirmed)\b
            """,
            re.IGNORECASE
        )
    ),

    # Trend language — directional words implying magnitude without evidence
    (
        "trend_language", "Medium",
        "Directional language — what is the actual magnitude and baseline?",
        re.compile(
            r"""(?x)
            \b(?:surged?|soared?|skyrocketed?|spiked?|jumped?|leaped?|shot\s+up|
               plummeted?|plunged?|collapsed?|cratered?|nosedived?|tanked?|
               slumped?|tumbled?|dropped?\s+sharply|fell?\s+sharply|
               rose?\s+sharply|rose?\s+dramatically|climbed?\s+sharply|
               declined?\s+sharply|declined?\s+dramatically|
               rapidly\s+(?:increased?|decreased?|grew?|fell?)|
               significantly\s+(?:increased?|decreased?|grew?|fell?|higher|lower|worse|better)|
               dramatically\s+(?:increased?|decreased?|rose?|fell?|dropped?|worse(?:ned?)?|deteriorated?)|
               dramatic\s+(?:drop|decline|fall|rise|increase)|
               escalated\s+sharply)
            \b""",
            re.IGNORECASE
        )
    ),

    # Comparative claims — superlatives and relative comparisons without reference
    (
        "comparative_claim", "Medium",
        "Comparative claim — compared to what, over what period?",
        re.compile(
            r"""(?x)
            \b(?:highest|lowest|most|least|best|worst|largest|smallest|
               greatest|fewest|fastest|slowest|first|last)\b
            | \bmore\s+than\b | \bless\s+than\b | \bfewer\s+than\b
            | \bat\s+(?:an?\s+)?all[-\s]time\b
            | \b(?:higher|lower|greater|smaller)\s+than\b
            | \bhighly\s+(?:unlikely|likely|specific|improbable)\b
            """,
            re.IGNORECASE
        )
    ),

    # Temporal claims — time references that may be imprecise or stale
    (
        "temporal_claim", "Medium",
        "Time reference — verify the period is accurate and current",
        re.compile(
            r"""(?x)
            \b(?:last|this|next)\s+(?:year|month|week|decade|quarter|fiscal\s+year)\b
            | \bsince\s+(?:19|20)\d{2}\b
            | \bin\s+(?:19|20)\d{2}\b
            | \bin\s+recent\s+(?:years?|months?|weeks?|decades?)\b
            | \bover\s+the\s+(?:past|last)\s+\d+\s+(?:years?|months?|decades?)\b
            | \bhistorically\b | \bfor\s+(?:decades?|years?|generations?)\b
            """,
            re.IGNORECASE
        )
    ),
]

### Tier 2 — spaCy patterns (dependency parse + NER, still deterministic)

In [16]:
# Causal connectives — phrases that assert causation
# Source: Penn Discourse Treebank causal relation markers
CAUSAL_CONNECTIVES = [
    "led to", "leads to", "lead to",
    "caused", "causes", "cause",
    "resulted in", "results in",
    "because of", "due to", "owing to",
    "triggered", "triggers",
    "drove", "drives",
    "produced", "produces",
    "contributed to", "contributes to",
    "as a result of", "as a consequence of",
]

# Certainty verbs — assert fact without hedge
# Contrast: "suggests", "indicates", "may" are hedged and not flagged
CERTAINTY_VERBS = [
    "shows", "show",
    "proves", "prove",
    "confirms", "confirm",
    "demonstrates", "demonstrate",
    "reveals", "reveal",
    "establishes", "establish",
    "means", "mean",
]

# spaCy NER labels and how to handle each
# "named" = always flag (must verify)
# "quantitative" = flag as quantitative_claim (numbers the regex missed)
# "temporal" = flag as temporal_claim (specific times/dates)
NER_RULES = {
    "PERSON":   ("named_entity",       "Medium", "PERSON — verify name and any claims attributed to them"),
    "ORG":      ("named_entity",       "Medium", "ORG — verify organization name and role"),
    "GPE":      ("named_entity",       "Medium", "GPE — verify place name and context"),
    "NORP":     ("named_entity",       "Medium", "NORP — verify group name and characterization"),
    "MONEY":    ("quantitative_claim", "High",   "Monetary amount — verify figure and source"),
    "CARDINAL": ("quantitative_claim", "High",   "Specific count — verify figure and source"),
    "PERCENT":  ("quantitative_claim", "High",   "Percentage — verify figure and source"),
    "DATE":     ("temporal_claim",     "Medium", "Date — verify accuracy and relevance"),
    "TIME":     ("temporal_claim",     "Medium", "Time — verify accuracy (exact times are high-risk in breaking news)"),
}


def _flag_spacy(doc) -> list[Flag]:
    """Run Tier 2 checks on a parsed spaCy doc."""
    flags = []
    text_lower = doc.text.lower()

    # Causal connectives — phrase match on lowercased text
    for phrase in CAUSAL_CONNECTIVES:
        for m in re.finditer(re.escape(phrase), text_lower):
            flags.append(Flag(
                start=m.start(), end=m.end(),
                text=doc.text[m.start():m.end()],
                flag_type="causal_claim",
                priority="High",
                reason="Asserts causation — verify mechanism and evidence"
            ))

    # Certainty verbs — token-level match
    for token in doc:
        if token.lemma_.lower() in CERTAINTY_VERBS and token.pos_ == "VERB":
            flags.append(Flag(
                start=token.idx, end=token.idx + len(token.text),
                text=token.text,
                flag_type="certainty_language",
                priority="Medium",
                reason="Certainty verb without hedge — consider 'suggests' or 'indicates'"
            ))

    # Named entities — type-specific handling via NER_RULES
    for ent in doc.ents:
        if ent.label_ in NER_RULES:
            flag_type, priority, reason = NER_RULES[ent.label_]
            flags.append(Flag(
                start=ent.start_char, end=ent.end_char,
                text=ent.text,
                flag_type=flag_type,
                priority=priority,
                reason=reason
            ))

    return flags

## 4. Core function: `flag_text()`

In [17]:
def flag_text(text: str) -> list[Flag]:
    """
    Main entry point.
    Returns a list of Flag objects with character offsets into `text`.
    Flags are sorted by position. Overlapping spans of the SAME flag type are
    deduplicated (keep highest priority). Different flag types on the same span
    are both kept — a phrase can be both a vague_attribution and contain a
    certainty_language verb; those are separate editorial concerns.
    """
    flags = []

    # Tier 1: regex
    for flag_type, priority, reason, pattern in REGEX_PATTERNS:
        for m in pattern.finditer(text):
            flags.append(Flag(
                start=m.start(), end=m.end(),
                text=m.group(),
                flag_type=flag_type,
                priority=priority,
                reason=reason
            ))

    # Tier 2: spaCy
    doc = nlp(text)
    flags.extend(_flag_spacy(doc))

    # Sort by position, then by priority (High first)
    priority_rank = {"High": 0, "Medium": 1, "Low": 2}
    flags.sort(key=lambda f: (f.start, priority_rank[f.priority]))

    # Deduplicate: only collapse overlapping spans of the same flag_type
    seen: dict[str, int] = {}  # flag_type -> index of last added flag of that type
    deduped = []
    for flag in flags:
        last_idx = seen.get(flag.flag_type)
        if last_idx is not None and flag.start < deduped[last_idx].end:
            # Same type, overlapping — keep higher priority (already sorted, so skip)
            pass
        else:
            seen[flag.flag_type] = len(deduped)
            deduped.append(flag)

    # Re-sort final list by position for rendering
    deduped.sort(key=lambda f: f.start)
    return deduped

## 5. Rendering — colored inline HTML

In [18]:
def _resolve_overlaps(flags: list[Flag]) -> list[tuple[Flag, list[Flag]]]:
    """
    For rendering, overlapping spans must not nest.
    Returns a list of (primary_flag, [all_flags_at_this_position]) tuples.
    Primary flag is the highest-priority one shown in the inline view.
    The full list is used to build multi-flag tooltips.
    """
    # Group all flags by position — find which ones overlap each primary span
    sorted_flags = sorted(flags, key=lambda f: (f.start, PRIORITY_RANK[f.priority]))
    resolved = []

    for flag in sorted_flags:
        if resolved and flag.start < resolved[-1][0].end:
            # Overlaps with last placed span — add to its co-flags, don't place separately
            resolved[-1][1].append(flag)
        else:
            resolved.append((flag, [flag]))

    return resolved


def _legend_html() -> str:
    """Inline color legend, one chip per flag type."""
    chips = " &nbsp; ".join(
        f'<span style="background:{color};padding:2px 7px;border-radius:3px;'
        f'font-size:11px;white-space:nowrap">{ftype.replace("_", " ")}</span>'
        for ftype, color in FLAG_COLORS.items()
    )
    return f'<p style="margin-top:10px;line-height:2">{chips}</p>'


def render_html(text: str, flags: list[Flag]) -> str:
    """
    Wraps flagged spans in <mark> tags colored by flag type.
    When multiple flags overlap the same span, the highest-priority color is shown
    and the tooltip lists all flag types at that position.
    A small indicator (*) marks multi-flag spans.
    """
    if not flags:
        return f"<div style='font-family:sans-serif'><p>{text}</p><em>No risk flags found.</em></div>"

    resolved = _resolve_overlaps(flags)
    parts = []
    cursor = 0

    for primary, co_flags in resolved:
        parts.append(text[cursor:primary.start])

        color = FLAG_COLORS.get(primary.flag_type, "#eeeeee")
        multi = len(co_flags) > 1

        if multi:
            # Build tooltip listing all co-flags
            lines = " | ".join(
                f"{f.flag_type} [{f.priority}]: {f.reason}" for f in co_flags
            )
            tooltip = f"MULTIPLE FLAGS — {lines}"
            # Underline indicates multi-flag span
            style = (
                f"background:{color};padding:2px 4px;border-radius:3px;"
                f"cursor:help;text-decoration:underline dotted #333;text-underline-offset:3px"
            )
            indicator = '<sup style="font-size:9px;color:#333">+</sup>'
        else:
            tooltip = f"{primary.flag_type} [{primary.priority}]: {primary.reason}"
            style = f"background:{color};padding:2px 4px;border-radius:3px;cursor:help"
            indicator = ""

        parts.append(
            f'<mark style="{style}" title="{tooltip}">'
            f'{primary.text}{indicator}</mark>'
        )
        cursor = primary.end

    parts.append(text[cursor:])
    body = "".join(parts)

    return f"""
    <div style="font-family:sans-serif;font-size:15px;line-height:1.8;max-width:820px">
        <p style="white-space:pre-wrap">{body}</p>
        {_legend_html()}
        <p style="font-size:11px;color:#999;margin-top:2px">
            Hover for details &nbsp;|&nbsp;
            <u style="text-decoration:underline dotted #333">underlined+</u> = multiple flag types on same span
        </p>
    </div>
    """


def render_table(flags: list[Flag]) -> str:
    """Returns an HTML table of all flags sorted by position.
    Shows all flags including those that overlap the same span."""
    if not flags:
        return "<p><em>No flags.</em></p>"

    rows = "".join(
        f"<tr>"
        f"<td style='padding:4px 8px'>"
        f"  <span style='background:{FLAG_COLORS.get(f.flag_type,'#eee')};padding:1px 6px;"
        f"border-radius:3px;font-size:11px'>{f.flag_type.replace('_',' ')}</span>"
        f"</td>"
        f"<td style='padding:4px 8px;font-size:12px;color:#555'>{f.priority}</td>"
        f"<td style='padding:4px 8px'><code>{f.text}</code></td>"
        f"<td style='padding:4px 8px;color:#555;font-size:12px'>{f.reason}</td>"
        f"</tr>"
        for f in flags
    )
    return f"""
    <table style="font-family:sans-serif;font-size:13px;border-collapse:collapse;width:100%">
        <thead>
            <tr style="background:#f0f0f0">
                <th style="padding:4px 8px;text-align:left">Flag type</th>
                <th style="padding:4px 8px;text-align:left">Priority</th>
                <th style="padding:4px 8px;text-align:left">Matched text</th>
                <th style="padding:4px 8px;text-align:left">Reason</th>
            </tr>
        </thead>
        <tbody>{rows}</tbody>
    </table>
    <p style="font-size:11px;color:#aaa;margin-top:4px">
        All flags shown — including multiple flags on the same span.
    </p>
    """

## 6. Run it

Paste any journalism text or AI output into `TEXT` below.

In [22]:
TEXT = """A group of new Polymarket accounts made highly specific, well-timed bets on whether the U.S. and Iran would reach a ceasefire on April 7, resulting in an estimated $2.1 million in theoretical profits.

These bets came even though, in the critical hours before the ceasefire was announced, President Donald Trump’s rhetoric had escalated sharply and there were very few signals that a ceasefire deal was imminent. Earlier in the day on Tuesday, President Trump had issued a warning on Truth Social that “a whole civilization will die tonight” if Iran did not meet his demands.

The analysis of publicly available blockchain data from Polymarket, using the crypto analytics platform Dune, shows that at least 50 newly created wallets with no prior trading history placed substantial “Yes” bets before President Donald Trump announced a two-week ceasefire at around 6:30 pm ET on Wednesday.

One of these wallets, created on April 7 around 2 pm ET, placed roughly $72,000 in bets at an average price of 8.8 cents, implying the market viewed a ceasefire as highly unlikely when those bets were made.
That wallet is estimated to have earned a profit of $766,000.

Another wallet, created 12 minutes before President Trump’s post on Truth Social, bought $31,908 in “yes” contracts, and is estimated to have earned a profit of $73,800.

Public blockchain data cannot identify who controls these wallets. Polymarket uses proxy smart contract wallets, meaning a single user can create multiple accounts. Only Polymarket has the internal data needed to determine whether these were new users or existing users opening additional accounts.

Polymarket did not immediately respond to a request for comment.

The profits earned by these users have not yet actualized. On Wednesday, Polymarket labeled the April 7 Iran-U.S. ceasefire contract as disputed due to the fact that Iran was still placing restrictions on the Strait of Hormuz as well as the ongoing missile attacks still occurring in the region. 

The trading pattern mirrors earlier episodes on Polymarket, including bets ahead of the January capture of Venezuelan President Nicolás Maduro, when newly created accounts placed large wagers hours before the operation became public and made hundreds of thousands of dollars in profit. Similar clusters of accounts have also repeatedly profited from well-timed bets on military actions involving Iran.

These well-timed bets have repeatedly raised questions from the public as well as Members of Congress about whether some traders are using insider information to profit in these prediction markets. Bipartisan groups of Senators as well as Representatives have introduced legislation that would broaden the definition of insider trading to include prediction markets. 

Even the two biggest platforms in the industry, Kalshi and Polymarket, have said they see a need to broaden the definition of insider trading on their platforms.

“This is why these markets need regulation,” said Todd Philips, a professor at Georgia State University who has written on prediction markets and the industry’s regulations. “We can’t have people trading market with inside information and expect other traders are going to be OK being in these markets."""

flags = flag_text(TEXT)
display(HTML(render_html(TEXT, flags)))

In [23]:
# Flag summary table
display(HTML(render_table(flags)))

Flag type,Priority,Matched text,Reason
comparative claim,Medium,highly specific,"Comparative claim — compared to what, over what period?"
named entity,Medium,U.S.,GPE — verify place name and context
named entity,Medium,Iran,GPE — verify place name and context
temporal claim,Medium,April 7,Date — verify accuracy and relevance
quantitative claim,High,an estimated $2.1 million,Hedged figure — does the reporter have the exact number?
temporal claim,Medium,hours,Time — verify accuracy (exact times are high-risk in breaking news)
named entity,Medium,Donald Trump,PERSON — verify name and any claims attributed to them
trend language,Medium,escalated sharply,Directional language — what is the actual magnitude and baseline?
temporal claim,Medium,Earlier in the day,Time — verify accuracy (exact times are high-risk in breaking news)
temporal claim,Medium,Tuesday,Date — verify accuracy and relevance


## 7. Quick tests

Spot-check that the core patterns fire as expected.  
These are not exhaustive — see `evaluation/` for the full gold set.

In [24]:
def assert_flag(text: str, expected_type: str):
    found = [f.flag_type for f in flag_text(text)]
    assert expected_type in found, (
        f"Expected '{expected_type}' in flags for: {text!r}\nGot: {found}"
    )
    print(f"  PASS  {expected_type:25s}  '{text.strip()}'")

def assert_reason_contains(text: str, flag_type: str, substr: str):
    flags = [f for f in flag_text(text) if f.flag_type == flag_type]
    assert flags, f"Expected '{flag_type}' flag for: {text!r}"
    assert any(substr in f.reason for f in flags), (
        f"Expected reason containing '{substr}' for '{flag_type}' in: {text!r}\n"
        f"Got reasons: {[f.reason for f in flags]}"
    )
    print(f"  PASS  {flag_type:25s}  reason='{substr}'  '{text.strip()}'")

def assert_no_flag(text: str, unexpected_type: str):
    found = [f.flag_type for f in flag_text(text)]
    assert unexpected_type not in found, (
        f"Expected NO '{unexpected_type}' for: {text!r}\nGot: {found}"
    )
    print(f"  PASS  no {unexpected_type:21s}  '{text.strip()}'")

print("--- Tier 1: Regex ---")
assert_flag("Evictions increased by 27%.", "quantitative_claim")
assert_flag("The city spent $4.2 million on housing.", "quantitative_claim")
assert_flag("$2.1 million in theoretical profits.", "quantitative_claim")
assert_flag("8.8 cents per contract.", "quantitative_claim")
assert_flag("Experts say rents are rising.", "vague_attribution")
assert_flag("Studies show that crime increased.", "vague_attribution")
assert_flag("Evictions surged last year.", "trend_language")
assert_flag("rhetoric had escalated sharply", "trend_language")
assert_flag("This is the highest rate on record.", "comparative_claim")
assert_flag("market viewed a ceasefire as highly unlikely", "comparative_claim")
assert_flag("Since 2020, rents have climbed.", "temporal_claim")

print("\n--- Tier 2: spaCy ---")
assert_flag("The policy led to higher evictions.", "causal_claim")
assert_flag("The data shows that rents rose.", "certainty_language")
assert_flag("Mayor Johnson announced the plan.", "named_entity")
assert_flag("50 newly created wallets placed bets.", "quantitative_claim")
assert_flag("at around 6:30 pm ET on Wednesday", "temporal_claim")
assert_flag("created on April 7 around 2 pm ET", "temporal_claim")
assert_flag("profit of $766,000", "quantitative_claim")

print("\n--- Passive attribution (AI pattern) ---")
assert_flag("It was found that rents had increased.", "passive_attribution")
assert_flag("It has been reported that the agency knew.", "passive_attribution")
assert_flag("It is estimated that 400 jobs were lost.", "passive_attribution")
assert_flag("It was alleged that the firm misled investors.", "passive_attribution")
assert_flag("The practice is widely believed to be effective.", "passive_attribution")
assert_flag("The memo has been widely reported in local outlets.", "passive_attribution")
assert_no_flag("The report was released on Wednesday.", "passive_attribution")  # specific actor
assert_no_flag("The committee found no evidence of fraud.", "passive_attribution")  # active voice

print("\n--- Hedged quantitative claims ---")
assert_reason_contains("roughly $72,000 in bets", "quantitative_claim", "Hedged figure")
assert_reason_contains("nearly half of all filings", "quantitative_claim", "Hedged figure")
assert_reason_contains("approximately 400 jobs", "quantitative_claim", "Hedged figure")
assert_reason_contains("about a third of respondents", "quantitative_claim", "Hedged figure")
assert_reason_contains("almost 9 cents per share", "quantitative_claim", "Hedged figure")
assert_reason_contains("an estimated 2 million people", "quantitative_claim", "Hedged figure")
# Precise numbers should NOT get the hedged reason
assert_reason_contains("$72,000 in bets", "quantitative_claim", "Specific number")

print("\n--- Should NOT flag ---")
assert_no_flag("The data suggests rents may be rising.", "certainty_language")
assert_no_flag('He called it "an unprecedented crisis".', "quote")

print("\nAll tests passed.")

--- Tier 1: Regex ---
  PASS  quantitative_claim         'Evictions increased by 27%.'
  PASS  quantitative_claim         'The city spent $4.2 million on housing.'
  PASS  quantitative_claim         '$2.1 million in theoretical profits.'
  PASS  quantitative_claim         '8.8 cents per contract.'
  PASS  vague_attribution          'Experts say rents are rising.'
  PASS  vague_attribution          'Studies show that crime increased.'
  PASS  trend_language             'Evictions surged last year.'
  PASS  trend_language             'rhetoric had escalated sharply'
  PASS  comparative_claim          'This is the highest rate on record.'
  PASS  comparative_claim          'market viewed a ceasefire as highly unlikely'
  PASS  temporal_claim             'Since 2020, rents have climbed.'

--- Tier 2: spaCy ---
  PASS  causal_claim               'The policy led to higher evictions.'
  PASS  certainty_language         'The data shows that rents rose.'
  PASS  named_entity               'Mayo